# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге.

Данный эксперимент - это второй шаг. Здесь агент должен научиться проходить ночной уровень. Поэтоум фокусно натаскиваем его на 5-ом уровне. При этом разбавляем его опыт уровнями 1 и 4, чтобы не разучивался.

Тут используем претрейнед `VisionHead` и `RenderHead` из `WorldModel`. 

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 5
    generation_ind = 0
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    ####
    
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.agent.parent = dict(agent=optuna_trial.suggest_categorical('agent.parent', [
        '18n_ppo_tr_frostbite_02:9',
        '18n_ppo_tr_frostbite_02:11',
        '18n_ppo_tr_frostbite_02:17',
    ])) 
    HP.agent.sequence_length = 4 # length observation chain agent incepts
    HP.agent.ob_shape = (1, 178, 152) 
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.vision_head = dict(grid=(6, 6), features_counts=(16, 32, 64, 128), is_trainable=False)
    HP.agent.action_plan_lv_encoding = 'separate'
    HP.agent.is_causal_action_plan = True
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.attention_backend = 'EFFICIENT_ATTENTION'
    HP.agent.prediction_target = 'next_obs'
    HP.agent.render_heads = dict(heads_count=4, is_trainable=False)
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level5_101:1', # bear level, night
    ]
    HP.video.capture_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'],
    ]
    HP.video.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1_101:1', 
        'com.develorium.neurolab.frostbite_ram:level4_101:1', # bear level, day
        'com.develorium.neurolab.frostbite_ram:level5_101:1', # bear level, night
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'three_lives', 'no_igloo'], # 0
            
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 1
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 2
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 3
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 4
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 5
    ]
    # 80% of time learn to pass level 5, 20% of time - keep old experience
    HP.ppo.rollout_env_stories = [
        '0,1;0', # for levels 1 and 4 - play an ordinary game
        '2;1:6', # for night level - use conditioning 
        '2;1:6', # to teach an agent 
        '2;1:6', # to enter blinking igloo
        '2;1:6', # ...
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256 
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    # HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    # HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

# Results
<TBD>

Результаты не столь впечатляющие, как в `17e_study_23.2b`. Скажем так, на четвёрочку. В `17e_study_23.2b` нашлись несколько агентов, котороые цельных пять уровней одолели, здесь только четыре. Но, можно поработать и с этим. Главная то беда в 17-ой серии была в том, что агент хорошо играет либо на 1-4 уровнях, либо на 5-6 и т.д. Если агенты 18-ой серии чуть тяжелее учатся, но зато умеют играть и там и тут, то это победа.

Чемпионы:
|агент|предок|
|---|---|
|18n_ppo_tr_frostbite_02:51|9|
|18n_ppo_tr_frostbite_02:94|11|
|18n_ppo_tr_frostbite_02:49|17|
|18n_ppo_tr_frostbite_02:83|17|
|18n_ppo_tr_frostbite_02:86|17|

<img src="./img/levels_passed.png">
<img src="./img/reward.png">
<img src="./img/episode_r.png">

**Выводы**
1) попробовать 3-ий шаг на чемпионах